In [42]:
import torch
import numpy as numpy
import os
import shutil
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [44]:
import torch
import sys
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

/home/undergrad/2026/ojonesma/Spring2026/EyeClassifierEnv/bin/python
2.11.0+cu130
13.0
False


## Efficient Net
#### Olivia Jones-Martin

In [45]:
from utilities import *

# setting path variables for later use
TRAINING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/a. Training Set"
VALIDATION_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/b. Validation Set"
TESTING_SET_PATH = "./A. RFMiD_All_Classes_Dataset/1. Original Images/c. Testing Set"

GROUND_TRUTHS_PATH = "./A. RFMiD_All_Classes_Dataset/2. GroundtruthsAllClasses"
'''
GROUND_TRUTHS_PATH can also point to the folder containing ground truths for all classes
The way my dataset is set up is that there's 3 folders, one for the images, one for the metadata for all classes,
and one for the metadata in which any disease with less than 10 instances get grouped into the label other
This can be modified depending on which set of metadata you're using
-Olivia Jones-Martin
'''

TRAINING_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "a. RFMiD_Training_Labels.csv")
VALIDATION_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "b. RFMiD_Validation_Labels.csv")
TEST_METADATA_PATH = os.path.join(GROUND_TRUTHS_PATH, "c. RFMiD_Testing_Labels.csv")

Loss function being used is [binary cross entropy with logits loss](https://docs.pytorch.org/docs/1.7.1/generated/torch.nn.BCEWithLogitsLoss.html#torch.nn.BCEWithLogitsLoss). Weights are calculated for the pos_weight parameter for the function.

In [46]:
# Get counts for each of the 
trainingMetadata = pd.read_csv(TRAINING_METADATA_PATH)
diseaseLabels = trainingMetadata.columns[1:]

# counts includes every class except for ID
counts = trainingMetadata.iloc[:, 1:].sum()
print("Disease Counts:")
print(counts)

# total amount of images and labels in the training set
image_count = trainingMetadata.shape[0]
label_count = counts.shape[0]
print(f"Image count: {image_count}")
print(f"Label count: {label_count}")

'''
If there is an instance where a disease appears 0 times, give a warning and
set the count value to one to prevent a divide by zero error when calculating
the weights
'''
for i in range(label_count):
    if counts.iloc[i] == 0:
        counts.iloc[i] = 1
        print(f"WARNING! Zero instances of {diseaseLabels[i]} in dataset")

'''
Calculate pos_weights for binary cross entropy with logits loss
pos weight for a class should be (negative counts of class)/(positive counts of class)
Documentation in reference 2
'''
pos_weights = [(image_count - count)/count for count in counts]
print("pos_weigths for BCEWithLogitsLoss")
print(pos_weights)

Disease Counts:
Disease_Risk    1519
DR               376
ARMD             100
MH               317
DN               138
MYA              101
BRVO              73
TSLN             186
ERM               14
LS                47
MS                15
CSR               37
ODC              282
CRVO              28
TV                 6
AH                16
ODP               65
ODE               58
ST                 5
AION              17
PT                11
RT                14
RS                43
CRS               32
EDN               15
RPEC              22
MHL               11
RP                 6
CWS                3
CB                 1
ODPM               0
PRH                2
MNF                3
HR                 0
CRAO               2
TD                 3
CME                4
PTCR               5
CF                 3
VH                 1
MCA                1
VS                 1
BRAO               2
PLQ                1
HPED               1
CL                 1
dtype: int64
Image

Load the images

In [47]:
# metadata for training dataset already loaded in previous parts, only need to load in validatoin and
# test metadata dataframes
validationMetadata = pd.read_csv(VALIDATION_METADATA_PATH)
testMetadata = pd.read_csv(TEST_METADATA_PATH)

'''
efficientnet_b5 uses images sizes of 456x456 (reference 3),
normilization used is RGB means of [0.485, 0.456, 0.406] and
standard deviations of [0.229, 0.224, 0.225]. means and std
deviation are given by reference 1 in section 5.2
'''
augmenter = dataAugmenter(image_size = (456, 456),
                          norm_mean=(0.485, 0.456, 0.406),
                          norm_std=(0.229, 0.224, 0.225),
                          useCutOut=True)

# Augmenter class used to provide transformations
training_dataset = FundusImageDataset(metadata=trainingMetadata,
                                      image_directory=TRAINING_SET_PATH,
                                      transform=augmenter.transform_train)
validation_dataset = FundusImageDataset(metadata=validationMetadata,
                                        image_directory=VALIDATION_METADATA_PATH,
                                        transform=augmenter.transform_test)
test_dataset = FundusImageDataset(metadata=testMetadata,
                                  image_directory=TESTING_SET_PATH,
                                  transform=augmenter.transform_test)

AttributeError: module 'torchvision.transforms' has no attribute 'compose'